# 04 Train and Evaluate with W&B

Run the full train-to-evaluation pipeline and log experiment results to Weights & Biases. This notebook logs config values, data sizes, train/validation losses, BLEU, chrF, sample translations, and checkpoint artifacts.

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

## W&B settings

Set `WANDB_MODE = "offline"` if you want to log locally without syncing immediately. For online logging, run `wandb login` before executing this notebook or set your W&B API key in the environment.

In [ ]:
PROJECT_NAME = "seq2seq-transformer"
RUN_NAME = "baseline"
WANDB_MODE = "online"  # "online" or "offline"
LOG_ARTIFACTS = True
NUM_SAMPLE_TRANSLATIONS = 10
WATCH_LOG = None  # None, "gradients", "parameters", or "all"
WATCH_LOG_FREQ = 100

os.environ["WANDB_MODE"] = WANDB_MODE

In [ ]:
import torch.optim as optim
import wandb

from src.config import Config
from src.data_pipeline import (
    create_dataloaders,
    extract_pairs,
    load_ko_en_dataset,
    maybe_take_subset,
    prepare_tokenizers,
    set_seed,
    split_pairs,
)
from src.evaluate import generate_predictions, print_sample_translations
from src.metrics import compute_bleu, compute_chrf
from src.model_utils import build_model
from src.train import create_loss_fn, save_checkpoint, train_one_epoch, validate_one_epoch
from src.wandb_experiment import config_for_wandb, log_checkpoint_artifacts

config = Config()
set_seed(config.random_seed)

print(f"device: {config.device}")
print(f"dataset: {config.dataset_name}")

## Initialize W&B run

In [ ]:
run = wandb.init(
    project=PROJECT_NAME,
    name=RUN_NAME,
    config=config_for_wandb(config),
)

run.name

## Load and split dataset

In [ ]:
dataset = load_ko_en_dataset(
    config.dataset_name,
    split=config.train_split,
    hf_token=config.hf_token,
)
pairs = extract_pairs(dataset, src_col="ko", tgt_col="en")

train_pairs, valid_pairs, test_pairs = split_pairs(
    pairs,
    valid_ratio=config.valid_ratio,
    test_ratio=config.test_ratio,
    seed=config.random_seed,
)

train_pairs = maybe_take_subset(train_pairs, config.train_subset_size)
valid_pairs = maybe_take_subset(valid_pairs, config.valid_subset_size)
test_pairs = maybe_take_subset(test_pairs, config.test_subset_size)

wandb.log({
    "data/total_pairs": len(pairs),
    "data/train_pairs": len(train_pairs),
    "data/valid_pairs": len(valid_pairs),
    "data/test_pairs": len(test_pairs),
})

print(f"total pairs: {len(pairs)}")
print(f"train pairs: {len(train_pairs)}")
print(f"valid pairs: {len(valid_pairs)}")
print(f"test pairs : {len(test_pairs)}")

## Prepare tokenizers and dataloaders

In [ ]:
sp_src, sp_tgt = prepare_tokenizers(train_pairs, config)

train_loader, valid_loader, test_loader = create_dataloaders(
    train_pairs,
    valid_pairs,
    test_pairs,
    sp_src,
    sp_tgt,
    config,
)

wandb.config.update(
    {
        "src_actual_vocab_size": sp_src.get_piece_size(),
        "tgt_actual_vocab_size": sp_tgt.get_piece_size(),
        "train_batches": len(train_loader),
        "valid_batches": len(valid_loader),
        "test_batches": len(test_loader),
    },
    allow_val_change=True,
)

print(f"src vocab size: {sp_src.get_piece_size()}")
print(f"tgt vocab size: {sp_tgt.get_piece_size()}")

## Build model

In [ ]:
model = build_model(
    config=config,
    src_vocab_size=sp_src.get_piece_size(),
    tgt_vocab_size=sp_tgt.get_piece_size(),
)
optimizer = optim.Adam(model.parameters(), lr=config.lr)
criterion = create_loss_fn(config.pad_id)

num_parameters = sum(p.numel() for p in model.parameters())
wandb.config.update({"num_parameters": num_parameters}, allow_val_change=True)

if WATCH_LOG is not None:
    wandb.watch(model, log=WATCH_LOG, log_freq=WATCH_LOG_FREQ)

print(f"parameters: {num_parameters:,}")

## Train and log losses

In [ ]:
history = []
best_valid_loss = float("inf")

for epoch in range(config.num_epochs):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        config.device,
    )
    valid_loss = validate_one_epoch(
        model,
        valid_loader,
        criterion,
        config.device,
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
    })

    wandb.log(
        {
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "valid/loss": valid_loss,
        },
        step=epoch + 1,
    )

    print(
        f"[Epoch {epoch + 1}/{config.num_epochs}] "
        f"train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f}"
    )

    latest_checkpoint_path = f"{config.checkpoint_dir}/latest.pt"
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        config=config,
        epoch=epoch + 1,
        train_loss=train_loss,
        valid_loss=valid_loss,
        src_vocab_size=sp_src.get_piece_size(),
        tgt_vocab_size=sp_tgt.get_piece_size(),
        path=latest_checkpoint_path,
    )

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_checkpoint_path = f"{config.checkpoint_dir}/best.pt"
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            config=config,
            epoch=epoch + 1,
            train_loss=train_loss,
            valid_loss=valid_loss,
            src_vocab_size=sp_src.get_piece_size(),
            tgt_vocab_size=sp_tgt.get_piece_size(),
            path=best_checkpoint_path,
        )
        wandb.run.summary["best_valid_loss"] = best_valid_loss
        wandb.run.summary["best_epoch"] = epoch + 1
        print(f"saved best checkpoint: {best_checkpoint_path}")

## Evaluate and log metrics

In [ ]:
source_texts, predictions, references = generate_predictions(
    model=model,
    dataloader=test_loader,
    sp_tgt=sp_tgt,
    config=config,
    device=config.device,
)

bleu = compute_bleu(predictions, references)
chrf = compute_chrf(predictions, references)

wandb.log({
    "test/bleu": bleu,
    "test/chrf": chrf,
})
wandb.run.summary["test_bleu"] = bleu
wandb.run.summary["test_chrf"] = chrf

print(f"BLEU: {bleu:.4f}")
print(f"chrF: {chrf:.4f}")

## Log sample translations

In [ ]:
sample_count = min(NUM_SAMPLE_TRANSLATIONS, len(predictions))
sample_table = wandb.Table(columns=["source", "reference", "prediction"])

for i in range(sample_count):
    sample_table.add_data(source_texts[i], references[i], predictions[i])

wandb.log({"samples/translations": sample_table})

print_sample_translations(
    source_texts=source_texts,
    predictions=predictions,
    references=references,
    n=sample_count,
)

## Log checkpoint artifacts and finish

In [ ]:
if LOG_ARTIFACTS:
    log_checkpoint_artifacts(wandb, config, run.name)

run.finish()